# Circuit Optimization Demo

Walks through every knob of `phasepoly_synthesize` (in [`src/phasepoly.py`](src/phasepoly.py)) on three benchmarks and verifies the outputs.

| Axis | Parameter | Values demoed |
|---|---|---|
| Synthesis method | `method` | `row_heap`, `pure_rotation_merging`, `rotation_merging` |
| Pre/post rotation merging | `rotation_merging_mode` | `pure_rotation_merging`, `advanced_rotation_merging` |
| Block grouping | `group_size` | `1` (single-block), `3` (cross-block) |
| Gaussian elimination | `gaussian_elim_algorithm` | `modified`, `classic` |
| Search frontier | `heap_size` | sweep `100 / 1000 / 10000` (default `10000`) |
| Search terminals | `ends_checked` | sweep `100 / 1000 / 10000` (default `10000`) |

Benchmarks: `adder_8`, `barenco_tof_10`, `ham15-med` (in `benchmarks/general/`). §1, §2, and §3a iterate over all three; §3b, §4, §5 focus on `adder_8`. The first cell is a one-line warm-up on `barenco_tof_4`.

Equivalence is checked at the end with `mqt.qcec` via `src.circuit_verification.verify_pair` (every circuit here exceeds the 8-qubit Qiskit cap, so that path auto-skips).

## Quickest run

Smallest call — self-contained (does its own imports and writes to `/tmp`).

In [3]:
import sys; sys.path.insert(0, ".")
from src.phasepoly import phasepoly_synthesize

r = phasepoly_synthesize(
    "benchmarks/general/barenco_tof_4.qasm", "/tmp/barenco_tof_4_opt.qasm",
    method="row_heap", heap_size=1000, ends_checked=1000, group_size=3,
)
print(f"Total Gates {r.input_circuit_info['gates(always weighted)']:>3} -> {r.circuit_info['gates(always weighted)']:>3}   "
    f"CX {r.input_circuit_info['weighted_cx']:>3} -> {r.circuit_info['weighted_cx']:>3}   "
      f"RZ {r.input_circuit_info['rz_gate']:>3} -> {r.circuit_info['rz_gate']:>3}   "
      f"{r.total_time:.2f}s")

Total Gates 114 ->  68   CX  48 ->  30   RZ  56 ->  28   2.69s


In [4]:
from pathlib import Path
import sys

# Resolve repo root so the notebook works whether launched from this folder or its parent.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    candidates = [c for c in PROJECT_ROOT.iterdir() if c.is_dir() and (c / "src" / "phasepoly.py").exists()]
    if not candidates:
        raise RuntimeError(f"Cannot locate PhasePoly src/ from {PROJECT_ROOT}")
    PROJECT_ROOT = candidates[0]
PROJECT_ROOT = PROJECT_ROOT.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.phasepoly import phasepoly_synthesize
from src.circuit_verification import verify_pair

BENCH_DIR = PROJECT_ROOT / "benchmarks" / "general"
OUT_DIR = PROJECT_ROOT / "results" / "demo_optimization"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CIRCUITS = {
    "adder_8":         BENCH_DIR / "adder_8.qasm",
    "barenco_tof_10":  BENCH_DIR / "barenco_tof_10.qasm",
    "ham15-med":       BENCH_DIR / "ham15-med.qasm",
}

for name, path in CIRCUITS.items():
    assert path.exists(), f"missing benchmark: {path}"
    print(f"{name:<16} -> {path.relative_to(PROJECT_ROOT)}")

adder_8          -> benchmarks/general/adder_8.qasm
barenco_tof_10   -> benchmarks/general/barenco_tof_10.qasm
ham15-med        -> benchmarks/general/ham15-med.qasm


In [5]:
RUN_RECORDS = []  # collected (circuit, label, input_path, output_path) tuples for verification at the end


def run_one(circuit, *, label, **kwargs):
    """Optimize one circuit and print a one-line summary. kwargs are forwarded to phasepoly_synthesize."""
    inp = CIRCUITS[circuit]
    out = OUT_DIR / f"{circuit}__{label}.qasm"
    res = phasepoly_synthesize(
        circuit_input_path=str(inp),
        circuit_output_path=str(out),
        circuit_name=circuit,
        label=label,
        **kwargs,
    )
    before = res.input_circuit_info
    after = res.circuit_info
    print(
        f"{circuit:<15} {label:<24} "
        f"Total Gates {before.get('gates(always weighted)', '?'):>5} -> {after.get('gates(always weighted)', '?'):>5}   "
        f"CX {before.get('weighted_cx', '?'):>4} -> {after.get('weighted_cx', '?'):>4}   "
        f"RZ {before.get('rz_gate', '?'):>4} -> {after.get('rz_gate', '?'):>4}   "
        f"{res.total_time:>5.1f}s"
    )
    RUN_RECORDS.append((circuit, label, str(inp), str(out)))
    return res

## Section 1 — Single-block baseline

Each phase-polynomial block (delimited by H gates) is optimized independently. This is the default mode (`group_size=1`) with the row-heap A* search and the modified Gaussian-elimination backend.

In [6]:
for name in CIRCUITS:
    run_one(
        name,
        label="s1_single_block",
        method="row_heap",
        rotation_merging_mode="advanced_rotation_merging",
        heap_size=1000,
        ends_checked=1000,
        group_size=1,
        gaussian_elim_algorithm="modified",
    )

adder_8         s1_single_block          Total Gates   900 ->   560   CX  409 ->  275   RZ  399 ->  207    32.8s
barenco_tof_10  s1_single_block          Total Gates   450 ->   262   CX  192 ->  128   RZ  224 ->  100     2.3s
ham15-med       s1_single_block          Total Gates  1272 ->   698   CX  534 ->  355   RZ  574 ->  253    53.8s


## Section 2 — Multi-block (cross-block) synthesis

`group_size > 1` merges adjacent phase-polynomial blocks into a combined parity table so the optimizer can reuse parity rows across H-gate barriers. Requires a `row_heap*` method.

In [7]:
for name in CIRCUITS:
    run_one(
        name,
        label="s2_multi_block_g3",
        method="row_heap",
        rotation_merging_mode="advanced_rotation_merging",
        heap_size=1000,
        ends_checked=1000,
        group_size=3,
        gaussian_elim_algorithm="modified",
    )

adder_8         s2_multi_block_g3        Total Gates   900 ->   550   CX  409 ->  265   RZ  399 ->  207    43.2s
barenco_tof_10  s2_multi_block_g3        Total Gates   450 ->   248   CX  192 ->  114   RZ  224 ->  100     8.7s
ham15-med       s2_multi_block_g3        Total Gates  1272 ->   693   CX  534 ->  350   RZ  574 ->  253   121.3s


## Section 3 — Pure rotation merging vs. advanced rotation merging

Rotation merging floats RZ rotations through CX gates and combines those that meet on the same parity. PhasePoly applies it twice (before and after synthesis) in two flavors:

- **`pure_rotation_merging`** — RZ floating only.
- **`advanced_rotation_merging`** — alternates RZ floating with `transform_cx_h_gates` (3 passes total). Often merges more rotations but takes longer.

§3a runs rotation merging *alone* (no phase-poly synthesis) on all three circuits — a merge-only baseline. §3b isolates `rotation_merging_mode` on `adder_8` with `method="row_heap"` fixed.

In [8]:
# 3a. Rotation-merging only (no synthesis): two methods compared on every circuit.
for name in CIRCUITS:
    run_one(name, label="s3a_pure_rm",     method="pure_rotation_merging")
    run_one(name, label="s3a_advanced_rm", method="rotation_merging")

adder_8         s3a_pure_rm              Total Gates   900 ->   716   CX  409 ->  409   RZ  399 ->  215     0.1s
adder_8         s3a_advanced_rm          Total Gates   900 ->   704   CX  409 ->  409   RZ  399 ->  207     0.1s
barenco_tof_10  s3a_pure_rm              Total Gates   450 ->   326   CX  192 ->  192   RZ  224 ->  100     0.0s
barenco_tof_10  s3a_advanced_rm          Total Gates   450 ->   326   CX  192 ->  192   RZ  224 ->  100     0.1s
ham15-med       s3a_pure_rm              Total Gates  1272 ->  1071   CX  534 ->  534   RZ  574 ->  373     0.1s
ham15-med       s3a_advanced_rm          Total Gates  1272 ->   897   CX  534 ->  534   RZ  574 ->  265     0.2s


In [9]:
# 3b. Same synthesis (row_heap), different rotation_merging_mode applied pre/post.
for mode in ("pure_rotation_merging", "advanced_rotation_merging"):
    run_one(
        "adder_8",
        label=f"s3b_rm-{mode}",
        method="row_heap",
        rotation_merging_mode=mode,
        heap_size=1000,
        ends_checked=1000,
        group_size=1,
        gaussian_elim_algorithm="modified",
    )

adder_8         s3b_rm-pure_rotation_merging Total Gates   900 ->   581   CX  409 ->  286   RZ  399 ->  215    32.3s
adder_8         s3b_rm-advanced_rotation_merging Total Gates   900 ->   560   CX  409 ->  275   RZ  399 ->  207    32.9s


## Section 4 — Gaussian-elimination backend: `modified` vs. `classic`

Inside the row-heap A\* search, every candidate parity row is reduced over GF(2):

- **`modified`** (default) — one-column lookahead pivot with delta-based diagonal repair.
- **`classic`** — historical greedy column-cost minimization.

Compared on `adder_8` with everything else held fixed.

In [10]:
for ge in ("modified", "classic"):
    run_one(
        "adder_8",
        label=f"s4_ge-{ge}",
        method="row_heap",
        rotation_merging_mode="advanced_rotation_merging",
        heap_size=1000,
        ends_checked=1000,
        group_size=1,
        gaussian_elim_algorithm=ge,
    )

adder_8         s4_ge-modified           Total Gates   900 ->   560   CX  409 ->  275   RZ  399 ->  207    33.6s
adder_8         s4_ge-classic            Total Gates   900 ->   562   CX  409 ->  277   RZ  399 ->  207    18.3s


## Section 5 — Heap and solution-size sweep

`heap_size` caps the A\* frontier; `ends_checked` caps the number of terminal states collected before returning the best. Sweeping both together at `100 / 1000 / 10000` on `adder_8` trades quality against runtime.

In [11]:
for size in (100, 1000, 10000):
    run_one(
        "adder_8",
        label=f"s5_heap-{size}",
        method="row_heap",
        rotation_merging_mode="advanced_rotation_merging",
        heap_size=size,
        ends_checked=size,
        group_size=1,
        gaussian_elim_algorithm="modified",
    )

adder_8         s5_heap-100              Total Gates   900 ->   562   CX  409 ->  277   RZ  399 ->  207     5.8s
adder_8         s5_heap-1000             Total Gates   900 ->   560   CX  409 ->  275   RZ  399 ->  207    33.9s
adder_8         s5_heap-10000            Total Gates   900 ->   557   CX  409 ->  272   RZ  399 ->  207   273.6s


## Section 6 — Equivalence verification

Every optimized circuit is checked against its input with `mqt.qcec`. All benchmarks here have more than 8 qubits, so the Qiskit `Operator.equiv` path is auto-skipped.

In [12]:
print(f"Verifying {len(RUN_RECORDS)} (input, output) pairs with mqt.qcec ... (timeout=90s/pair)\n")
fail = 0
for circuit, label, inp, out in RUN_RECORDS:
    rec = verify_pair(inp, out, methods=["qcec"], timeout=90)
    r = rec["qcec"]
    detail = str(r["detail"])
    ok = (r["status"] == "ok") and ("equivalent" in detail.lower()) and ("non" not in detail.lower())
    mark = "OK " if ok else f"{r['status'].upper():<8}"
    if not ok:
        fail += 1
    print(f"[{mark}] {circuit:<10} {label:<28} {detail[:48]:<48}  ({r['elapsed']:>5.1f}s)")
print(f"\n{len(RUN_RECORDS) - fail}/{len(RUN_RECORDS)} pairs verified equivalent.")

Verifying 19 (input, output) pairs with mqt.qcec ... (timeout=90s/pair)

[OK ] adder_8    s1_single_block              EquivalenceCriterion.equivalent_up_to_global_pha  (  0.3s)
[OK ] barenco_tof_10 s1_single_block              EquivalenceCriterion.equivalent                   ( 52.4s)
[OK ] ham15-med  s1_single_block              EquivalenceCriterion.equivalent                   (  0.3s)
[OK ] adder_8    s2_multi_block_g3            EquivalenceCriterion.equivalent                   (  0.3s)
[OK ] barenco_tof_10 s2_multi_block_g3            EquivalenceCriterion.equivalent                   ( 53.1s)
[OK ] ham15-med  s2_multi_block_g3            EquivalenceCriterion.equivalent                   (  0.3s)
[OK ] adder_8    s3a_pure_rm                  EquivalenceCriterion.equivalent                   (  0.3s)
[OK ] adder_8    s3a_advanced_rm              EquivalenceCriterion.equivalent                   (  0.3s)
[OK ] barenco_tof_10 s3a_pure_rm                  EquivalenceCriterion.equivale